# MiniMax H3 — ComfyUI + Pinggy

Stable fallback notebook for the original Pinggy-based public ComfyUI link.

Each Colab runtime generates a fresh temporary `*.pinggy.link` URL.


## 0. Mount Drive + persistence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PERSIST_MODELS_TO_DRIVE = False
PERSIST_OUTPUT_TO_DRIVE = True
DRIVE_ROOT = '/content/drive/MyDrive/MiniMax_H3_ComfyUI'


## 1. Check GPU

In [ ]:
import subprocess
def sh(cmd): return subprocess.check_output(cmd, shell=True, text=True).strip()
gpu_name = sh('nvidia-smi --query-gpu=name --format=csv,noheader | head -n1')
vram_mb = int(sh('nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1'))
name = gpu_name.lower()
H3_RESERVE_VRAM_GB = '6' if ('rtx pro 6000' in name or 'blackwell' in name) else '4' if 'a100' in name else '2' if 'l4' in name else '1'
print(f'GPU: {gpu_name} ({vram_mb/1024:.1f} GB)')


## 2. Clone preserved Pinggy branch

In [ ]:
%cd /content
!rm -rf /content/All-testing /content/minimax_h3_comfy
!git clone --depth 1 --branch minimax-h3-pinggy https://github.com/Logan17de/All-testing.git /content/All-testing
!cp -r /content/All-testing/video/minimax_h3_comfy /content/minimax_h3_comfy
%cd /content/minimax_h3_comfy


## 3. Install/update ComfyUI + H3 custom nodes

In [ ]:
import os, subprocess, sys
os.environ['COMFY_ROOT']='/content/ComfyUI'
os.environ['H3_DRIVE_ROOT']=DRIVE_ROOT
os.environ['H3_PERSIST_MODELS']='1' if PERSIST_MODELS_TO_DRIVE else '0'
os.environ['H3_PERSIST_OUTPUT']='1' if PERSIST_OUTPUT_TO_DRIVE else '0'
os.environ['H3_VRAM_MODE']='auto'
os.environ['H3_RESERVE_VRAM_GB']=H3_RESERVE_VRAM_GB
os.environ['H3_PREVIEW_METHOD']='none'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
!bash install_comfy_h3.sh
CUSTOM='/content/ComfyUI/custom_nodes'
def clone_or_pull(url, folder):
    path=f'{CUSTOM}/{folder}'
    if os.path.isdir(path+'/.git'): subprocess.run(['git','-C',path,'pull','--ff-only'], check=False)
    else: subprocess.run(['git','clone','--depth','1',url,path], check=True)
    req=os.path.join(path,'requirements.txt')
    if os.path.exists(req): subprocess.run([sys.executable,'-m','pip','install','-r',req], check=False)
clone_or_pull('https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git','ComfyUI_MiniMaxH3_Director')
clone_or_pull('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git','ComfyUI-VideoHelperSuite')
clone_or_pull('https://github.com/kijai/ComfyUI-KJNodes.git','ComfyUI-KJNodes')
clone_or_pull('https://github.com/pixaroma/ComfyUI-Pixaroma.git','ComfyUI-Pixaroma')


## 4. Download H3 models

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
MODEL_ROOT=Path('/content/ComfyUI/models')
for folder in ['diffusion_models','text_encoders','vae','loras','latent_upscale_models']: (MODEL_ROOT/folder).mkdir(parents=True, exist_ok=True)
downloads=[('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors',MODEL_ROOT,MODEL_ROOT/'diffusion_models'/'minimax_h3_fl2va_pruned_int8_convrot.safetensors'),('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors',MODEL_ROOT,MODEL_ROOT/'diffusion_models'/'minimax_h3_ref2va_pruned_int8_convrot.safetensors'),('Comfy-Org/MiniMax-H3','text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',MODEL_ROOT,MODEL_ROOT/'text_encoders'/'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'),('Comfy-Org/MiniMax-H3','vae/minimax_h3_video_vae_fp16.safetensors',MODEL_ROOT,MODEL_ROOT/'vae'/'minimax_h3_video_vae_fp16.safetensors'),('Comfy-Org/MiniMax-H3','vae/minimax_h3_audio_vae_fp32.safetensors',MODEL_ROOT,MODEL_ROOT/'vae'/'minimax_h3_audio_vae_fp32.safetensors'),('lightx2v/Minimax-h3-Turbo','minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors',MODEL_ROOT/'loras',MODEL_ROOT/'loras'/'minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors'),('lightx2v/Minimax-h3-Turbo','minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors',MODEL_ROOT/'loras',MODEL_ROOT/'loras'/'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors'),('LBH-123-AI/Minimax_h3_latent_Upscaler','minimax_h3_latent_upscaler_3d_fp16.safetensors',MODEL_ROOT/'latent_upscale_models',MODEL_ROOT/'latent_upscale_models'/'minimax_h3_latent_upscaler_3d_fp16.safetensors')]
for repo_id,filename,local_dir,target in downloads:
    if target.exists() and target.stat().st_size>1024*1024: print('SKIP',target.name); continue
    print('DOWNLOAD',filename); hf_hub_download(repo_id=repo_id,filename=filename,local_dir=str(local_dir))


## 5. Launch ComfyUI + Pinggy

The final output prints the fresh Pinggy URL.

In [ ]:
!bash launch_comfy.sh
